In [2]:
# generate_models.py
import pandas as pd
import numpy as np
import pickle
import os

# Algorithms
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# Metrics
from sklearn.metrics import (accuracy_score, roc_auc_score, precision_score, 
                             recall_score, f1_score, matthews_corrcoef)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. Initialize Folders
if not os.path.exists('model'):
    os.makedirs('model')

# 2. Load Data
# UCI dataset often has header issues. We skip row 0 if it contains ID info.
try:
    credit_df = pd.read_csv('default_of_credit_card_clients.csv', header=1)
except FileNotFoundError:
    print("Error: Please upload 'default_of_credit_card_clients.csv'")
    exit()

# Rename target for clarity
credit_df.rename(columns={'default payment next month': 'IsDefaulter'}, inplace=True)

# Basic Cleaning
if 'ID' in credit_df.columns:
    credit_df = credit_df.drop('ID', axis=1)

# Features (X) and Target (y)
X_data = credit_df.drop('IsDefaulter', axis=1)
y_data = credit_df['IsDefaulter']

# Scaling (Essential for KNN and Logistic Regression)
std_scaler = StandardScaler()
X_scaled = std_scaler.fit_transform(X_data)

# Split Data (80% Train, 20% Test)
X_trn, X_tst, y_trn, y_tst = train_test_split(X_scaled, y_data, test_size=0.2, random_state=42)

# Save Scaler for the App (Do not skip this!)
pickle.dump(std_scaler, open('model/scaler_custom.pkl', 'wb'))

# 3. Model Definitions
classifiers = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(max_depth=10),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Naive Bayes": GaussianNB(),
    "Random Forest": RandomForestClassifier(n_estimators=20, max_depth=10, n_jobs=-1),
    "XGBoost": XGBClassifier(n_estimators=20, max_depth=6, eval_metric='logloss')
}

# 4. Training and Evaluation Loop
performance_log = []

print("Training initiated... please wait.\n")

for clf_name, clf_model in classifiers.items():
    # Fit model
    clf_model.fit(X_trn, y_trn)
    
    # Predict
    preds = clf_model.predict(X_tst)
    probs = clf_model.predict_proba(X_tst)[:, 1] # Probability for AUC
    
    # Calculate Metrics [cite: 40-46]
    metrics = {
        "ML Model Name": clf_name,
        "Accuracy": round(accuracy_score(y_tst, preds), 4),
        "AUC": round(roc_auc_score(y_tst, probs), 4),
        "Precision": round(precision_score(y_tst, preds, zero_division=0), 4),
        "Recall": round(recall_score(y_tst, preds), 4),
        "F1": round(f1_score(y_tst, preds), 4),
        "MCC": round(matthews_corrcoef(y_tst, preds), 4)
    }
    performance_log.append(metrics)
    
    # Save Model File
    filename = f'model/{clf_name.replace(" ", "_").lower()}.pkl'
    pickle.dump(clf_model, open(filename, 'wb'))
    print(f"Saved: {filename}")

# 5. Display Comparison Table
results_table = pd.DataFrame(performance_log)

print("\n" + "="*50)
print("FINAL RESULTS TABLE (COPY FOR README)")
print("="*50)
print(results_table.to_markdown(index=False))

# Optional: Save to CSV
results_table.to_csv('final_metrics.csv', index=False)

Training initiated... please wait.

Saved: model/logistic_regression.pkl
Saved: model/decision_tree.pkl
Saved: model/knn.pkl
Saved: model/naive_bayes.pkl
Saved: model/random_forest.pkl
Saved: model/xgboost.pkl

FINAL RESULTS TABLE (COPY FOR README)
| ML Model Name       |   Accuracy |    AUC |   Precision |   Recall |     F1 |    MCC |
|:--------------------|-----------:|-------:|------------:|---------:|-------:|-------:|
| Logistic Regression |     0.8097 | 0.727  |      0.6913 |   0.2353 | 0.3511 | 0.3242 |
| Decision Tree       |     0.8108 | 0.7357 |      0.6206 |   0.3488 | 0.4466 | 0.3639 |
| KNN                 |     0.7952 | 0.708  |      0.5493 |   0.3564 | 0.4323 | 0.3252 |
| Naive Bayes         |     0.707  | 0.7371 |      0.3967 |   0.6504 | 0.4928 | 0.3218 |
| Random Forest       |     0.8213 | 0.7733 |      0.6754 |   0.3534 | 0.464  | 0.3971 |
| XGBoost             |     0.8183 | 0.7826 |      0.6538 |   0.361  | 0.4652 | 0.39   |
